# Simulador 17.2 — Optimizacion bajo Restricciones del Programa
## Modelo de Staddon: 3 comportamientos (trabajo, consumo, ocio)

Aprendizaje y Comportamiento Adaptable — Capitulo 17

---

MODELO COMPLETO

Funcion de costo con tres comportamientos:

  C(R) = a*(R0-R)^2  +  b*(r0-f(R))^2  +  c*(L0-L)^2

El termino de ocio simplifica a  c*((R0-R)+(r0-f(R)))^2  porque
L0 - L = (R0-R) + (r0-f(R)) por la restriccion de tiempo total.

Parametros:
  a = peso costo de trabajo   (pequeno: el animal trabaja poco libremente)
  b = peso costo de consumo   (mayor: consumo muy valorado)
  c = peso costo de ocio      ** c=0 produce funcion circular; c>0 la elonga **

Defaults: R0=3, r0=8, a=0.1, b=1.0, c=0  (roedor tipico de laboratorio)

---

PANELES

Panel izquierdo: geometria para el programa y valor actuales.
Elipses de indiferencia + restriccion del programa + punto optimo (estrella).

Panel derecho: puntos que el estudiante agrega manualmente.
Al pulsar "Curva teorica completa" aparecen DOS curvas:
  -- linea punteada gris: c=0  (modelo de 2 comportamientos, circular)
  -- linea solida azul:   c actual (modelo completo, elongada)

---

PASOS

1. Elige RV o IV.
2. Mueve el slider al primer valor a explorar.
3. Observa la geometria en el panel izquierdo.
4. Clic en "Agregar punto".
5. Repite con varios valores para construir la curva.
6. Clic en "Curva teorica completa" para comparar.
7. Cambia c y observa como varia la forma de la curva.

RV sugeridos: 0.3, 0.5, 1, 2, 3, 5, 8, 12, 20
IV sugeridos: 0.5, 1, 2, 4, 6, 8, 10, 14


## Guia de exploracion

BASICAS
- Con RV y c=0, construye la funcion con los valores sugeridos.
  Es bitonicas? En que valor del programa esta el pico de R*?

- Ahora aumenta c a 0.5, pulsa "Curva teorica completa".
  Que le ocurre al pico y a la rama descendente?
  Por que se dice que la funcion se "elonga"?

- Cambia a IV con c=0. Construye la funcion.
  Por que NO es bitonicas bajo IV? Pista: observa la forma de la restriccion.

INTERMEDIAS
- Con c=0 la funcion es aproximadamente circular (simetrica).
  Con c=1.5 la rama descendente es mucho mas gradual.
  Explica por que: cuando R > R0 y r < r0, los dos terminos en ((R0-R)+(r0-r))
  tienen signos opuestos y se cancelan parcialmente. Que implica eso para el costo?

- Aumenta a de 0.1 a 0.8. Como cambia la funcion? Que dice un a grande sobre
  las preferencias libres del organismo?

- Con RV, mueve R0 a 8.0. La funcion sigue siendo bitonicas? Por que cambia?

AVANZADA
- Compara las curvas teoricas con c=0 y c=2.0. Coinciden en algun punto especifico?
  Que tiene de especial ese punto?

- Usando el panel izquierdo con programa IV, explica geometricamente por que
  el punto optimo sobre una restriccion concava implica igualar las tasas
  relativas de respuesta y de refuerzo (igualacion en programas concurrentes).


In [ ]:
# !pip install -q ipywidgets
# from google.colab import output
# output.enable_custom_widget_manager()

import warnings
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.optimize import minimize_scalar

AZUL    = '#2C5282'
NARANJA = '#C05621'
VERDE   = '#276749'
GRIS    = '#718096'

matplotlib.rcParams['font.family'] = 'DejaVu Serif'

K_IV = 20.0          # parametro de forma de la restriccion IV
state = {'ns': [], 'Rs': []}   # puntos acumulados por el estudiante

# ── Funciones de retroalimentacion ───────────────────────────────────────────
def f_RV(R, n):
    return R / n

def f_IV(R, lam):
    return lam * R / (R + K_IV)

def get_fb(programa, val):
    return (lambda R: f_RV(R, val)) if programa == 'RV' else (lambda R: f_IV(R, val))


# ── Modelo completo de Staddon (3 comportamientos) ────────────────────────────
# Costo = a*(R0-R)^2 + b*(r0-f(R))^2 + c*(L0-L)^2
# Con L = T - R - f(R),  L0 = T - R0 - r0, el ultimo termino simplifica a:
# c * ( (R0-R) + (r0-f(R)) )^2
# -- no requiere escalar L por separado --

def costo_3B(R, R0, r0, a, b, c, fb):
    dR = R0 - R
    dr = r0 - fb(R)
    return a*dR**2 + b*dr**2 + c*(dR + dr)**2

def punto_optimo(R0, r0, a, b, c, programa, val):
    fb = get_fb(programa, val)
    res = minimize_scalar(
        lambda R: costo_3B(R, R0, r0, a, b, c, fb),
        bounds=(0.001, max(R0*5, 70.0)),
        method='bounded')
    return res.x, fb(res.x)


def funcion_completa(R0, r0, a, b, c, programa):
    ns = np.linspace(0.3, 35.0, 300) if programa == 'RV' else np.linspace(0.5, 14.0, 300)
    Rs = np.array([punto_optimo(R0, r0, a, b, c, programa, n)[0] for n in ns])
    return ns, Rs


# ── Widgets ───────────────────────────────────────────────────────────────────
WS = dict(style={'description_width': '130px'}, layout=widgets.Layout(width='340px'))

prog_drop = widgets.Dropdown(
    options=['RV', 'IV'], value='RV',
    description='Tipo de programa:',
    style={'description_width': '145px'},
    layout=widgets.Layout(width='310px'))

n_slider  = widgets.FloatSlider(
    min=0.3, max=20.0, step=0.3, value=1.0,
    description='Valor / Tasa ref.:', **WS)
R0_slider = widgets.FloatSlider(min=0.5, max=12.0, step=0.5, value=3.0,
                                 description='R0 trabajo pref.:', **WS)
r0_slider = widgets.FloatSlider(min=2.0,  max=14.0, step=0.5, value=8.0,
                                 description='r0 consumo pref.:', **WS)
a_slider  = widgets.FloatSlider(min=0.05, max=1.0,  step=0.05, value=0.1,
                                 description='peso a (trabajo):', **WS)
b_slider  = widgets.FloatSlider(min=0.25, max=4.0,  step=0.25, value=1.0,
                                 description='peso b (consumo):', **WS)
c_slider  = widgets.FloatSlider(min=0.0,  max=3.0,  step=0.25, value=0.0,
                                 description='peso c (ocio):', **WS)

btn_add   = widgets.Button(description='Agregar punto',
                            button_style='success', icon='plus',
                            layout=widgets.Layout(width='155px'))
btn_clear = widgets.Button(description='Limpiar',
                            button_style='danger',  icon='trash',
                            layout=widgets.Layout(width='115px'))
btn_full  = widgets.Button(description='Curva teorica completa',
                            button_style='info', icon='line-chart',
                            layout=widgets.Layout(width='215px'))

out_left  = widgets.Output()
out_right = widgets.Output()


# ── Panel izquierdo: geometria ────────────────────────────────────────────────
def draw_left(programa, n_val, R0, r0, a, b, c):
    LMAX = 24
    Rg = np.linspace(0.02, LMAX, 340)
    rg = np.linspace(0.02, LMAX, 340)
    RR, rr = np.meshgrid(Rg, rg)

    U_q   = -(a*(RR-R0)**2 + b*(rr-r0)**2 + c*((R0-RR)+(r0-rr))**2)
    U_min = float(np.nanmin(U_q))
    levels = np.linspace(U_min*0.88, U_min*0.04, 9)

    R_line = np.linspace(0, LMAX, 500)
    fb     = get_fb(programa, n_val)
    r_c    = np.clip(np.array([fb(R) for R in R_line]), 0, LMAX)
    R_star, r_star = punto_optimo(R0, r0, a, b, c, programa, n_val)

    fig, ax = plt.subplots(figsize=(5.5, 5.2), facecolor='white')
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible(False)

    ax.contour(RR, rr, U_q, levels=levels,
               colors=[AZUL]*9, linewidths=1.3, alpha=0.55)
    ax.plot(R0, r0, 'o', color=NARANJA, markersize=11, zorder=6,
            label='B0=({:.0f},{:.0f})'.format(R0, r0))

    color_c = NARANJA if programa == 'RV' else VERDE
    lbl_c   = 'Restriccion lineal (RV)' if programa == 'RV' else 'Restriccion concava (IV)'
    ax.plot(R_line, r_c, color=color_c, lw=2.5, label=lbl_c, zorder=5)

    color_star = VERDE if programa == 'RV' else NARANJA
    ax.plot(R_star, r_star, '*', color=color_star, markersize=15, zorder=7,
            label='R*={:.2f},  r*={:.2f}'.format(R_star, r_star))
    ax.plot([0, R_star], [r_star, r_star], '--', color=GRIS, lw=0.8, alpha=0.6)
    ax.plot([R_star, R_star], [0, r_star], '--', color=GRIS, lw=0.8, alpha=0.6)

    costo_actual = costo_3B(R_star, R0, r0, a, b, c, fb)
    ax.text(0.02, 0.97, 'Costo min.: {:.2f}'.format(costo_actual),
            transform=ax.transAxes, va='top', fontsize=9, color=GRIS,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=GRIS, alpha=0.9))
    ax.text(0.98, 0.04,
            ('Restriccion lineal\n(razon variable)' if programa == 'RV'
             else 'Restriccion concava\n(intervalo variable)'),
            transform=ax.transAxes, ha='right', fontsize=8, color=color_c,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color_c, alpha=0.9))

    ax.set_xlim(0, LMAX); ax.set_ylim(0, LMAX)
    ax.set_xlabel('Tasa de trabajo  R  (resp/min)', fontsize=10)
    ax.set_ylabel('Tasa de consumo  r  (ref/min)', fontsize=10)
    ax.set_title('Geometria: curvas de indiferencia\ny restriccion del programa\n(Staddon — 3 comportamientos)',
                 fontsize=10, color=AZUL)
    ax.grid(True, alpha=0.18, color=GRIS)
    ax.legend(fontsize=8.5, loc='upper right')
    plt.tight_layout()
    plt.show()


# ── Panel derecho: funcion de respuesta ───────────────────────────────────────
def draw_right(programa, show_theory=False, R0=3, r0=8, a=0.1, b=1.0, c=0.0):
    ns_arr = np.array(state['ns'])
    Rs_arr = np.array(state['Rs'])

    if programa == 'RV':
        xlabel = 'Valor del programa RV  (resp/ref)\nMayor = mas exigente'
        titulo = 'Funcion de respuesta  RV'
    else:
        xlabel = 'Tasa de refuerzo programada  (ref/min)\nMayor = mas rico'
        titulo = 'Funcion de respuesta  IV'

    fig, ax = plt.subplots(figsize=(5.5, 5.2), facecolor='white')
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible(False)

    if show_theory:
        # Curva de referencia: c=0 (dos comportamientos, funcion circular)
        if programa == 'RV':
            ns_ref, Rs_ref = funcion_completa(R0, r0, a, b, 0.0, programa)
            ax.plot(ns_ref, Rs_ref, color=GRIS, lw=1.8, ls='--', alpha=0.7,
                    label='c=0  (2 comportamientos, circular)')
        # Curva completa con ocio (c actual)
        ns_t, Rs_t = funcion_completa(R0, r0, a, b, c, programa)
        lbl = 'c={:.2f}  (3 comportamientos, elongada)'.format(c) if c > 0 else 'c=0  (2 comportamientos)'
        ax.plot(ns_t, Rs_t, color=AZUL, lw=2.5, label=lbl)

    if len(ns_arr) > 0:
        idx = np.argsort(ns_arr)
        ax.plot(ns_arr[idx], Rs_arr[idx], 'o-', color=NARANJA,
                lw=2.0, markersize=9, zorder=5, label='Tus puntos')
        for n_, r_ in zip(ns_arr[idx], Rs_arr[idx]):
            ax.annotate('{:.1f}'.format(n_), xy=(n_, r_),
                        xytext=(0, 8), textcoords='offset points',
                        fontsize=7.5, color=GRIS, ha='center')
    elif not show_theory:
        ax.text(0.5, 0.5,
                'Agrega puntos con\n"Agregar punto"',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=12, color=GRIS, style='italic')

    ax.set_xlabel(xlabel, fontsize=9.5)
    ax.set_ylabel('Tasa de respuesta optima  R*  (resp/min)', fontsize=10)
    ax.set_title(titulo + '\n({} puntos)'.format(len(ns_arr)),
                 fontsize=10, color=AZUL)
    ax.grid(True, alpha=0.18, color=GRIS)
    if show_theory or len(ns_arr) > 0:
        ax.legend(fontsize=8.5)
    plt.tight_layout()
    plt.show()


# ── Callbacks ─────────────────────────────────────────────────────────────────
def _params():
    return (prog_drop.value, n_slider.value,
            R0_slider.value, r0_slider.value,
            a_slider.value, b_slider.value, c_slider.value)

def refresh_left(change=None):
    p, n, R0, r0, a, b, c = _params()
    with out_left:
        clear_output(wait=True)
        draw_left(p, n, R0, r0, a, b, c)

def on_add(btn):
    p, n, R0, r0, a, b, c = _params()
    R_star, _ = punto_optimo(R0, r0, a, b, c, p, n)
    state['ns'].append(n)
    state['Rs'].append(R_star)
    with out_right:
        clear_output(wait=True)
        draw_right(p, show_theory=False, R0=R0, r0=r0, a=a, b=b, c=c)

def on_clear(btn):
    state['ns'].clear(); state['Rs'].clear()
    p, n, R0, r0, a, b, c = _params()
    with out_right:
        clear_output(wait=True)
        draw_right(p, R0=R0, r0=r0, a=a, b=b, c=c)

def on_full(btn):
    p, n, R0, r0, a, b, c = _params()
    with out_right:
        clear_output(wait=True)
        draw_right(p, show_theory=True, R0=R0, r0=r0, a=a, b=b, c=c)

for w in [prog_drop, n_slider, R0_slider, r0_slider, a_slider, b_slider, c_slider]:
    w.observe(refresh_left, names='value')
btn_add.on_click(on_add)
btn_clear.on_click(on_clear)
btn_full.on_click(on_full)


# ── Interfaz ──────────────────────────────────────────────────────────────────
titulo_lbl = widgets.HTML(
    '<h3 style="color:#2C5282;font-family:serif;">'
    'Optimizacion en Equilibrio &mdash; Modelo de Staddon (3 comportamientos)</h3>')

instruc_lbl = widgets.HTML(
    '<div style="font-family:serif;font-size:13px;color:#718096;">'
    '<b>Uso:</b> (1) Elige programa. '
    '(2) Mueve el slider al valor a explorar. '
    '(3) Clic en <b>Agregar punto</b>. '
    '(4) Repite para construir la curva. '
    '(5) Clic en <b>Curva teorica completa</b> para comparar '
    '(muestra c=0 punteado vs. c actual solido).'
    '</div>')

note_lbl = widgets.HTML(
    '<div style="font-family:serif;font-size:12px;color:#C05621;">'
    '<b>RV sugeridos:</b> 0.3, 0.5, 1, 2, 3, 5, 8, 12, 20 &nbsp;|&nbsp; '
    '<b>IV sugeridos:</b> 0.5, 1, 2, 4, 6, 8, 10, 14 &nbsp;|&nbsp; '
    '<b>Clave:</b> c=0 da funcion circular; c&gt;0 la <em>elonga</em>.'
    '</div>')

row_a = widgets.HBox([prog_drop,  n_slider])
row_b = widgets.HBox([R0_slider,  r0_slider])
row_c = widgets.HBox([a_slider,   b_slider])
row_d = widgets.HBox([c_slider])

btns_row = widgets.HBox([btn_add, btn_clear, btn_full],
                         layout=widgets.Layout(margin='6px 0 8px 0'))
fig_row  = widgets.HBox([out_left, out_right])

ui = widgets.VBox([titulo_lbl, instruc_lbl, note_lbl,
                   row_a, row_b, row_c, row_d, btns_row,
                   fig_row])
display(ui)

refresh_left()
with out_right:
    draw_right('RV')
